In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent

sys.path.insert(0, str(ROOT / "results" / "evaluation" ))

### *Functions*

##### Correlation

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, pearsonr
import matplotlib.pyplot as plt
import seaborn as sns


def get_corr_matrix(df, method="pearson"):
    corr = df.drop(columns=["name"]).corr(method=method)
    return corr


def get_p_values(df, method="pearson"):
  cols = df.columns
  p_val_df = pd.DataFrame(np.zeros((len(cols), len(cols))), index=cols, columns=cols)

  for i in range(len(cols)):
    for j in range(len(cols)):
      if i == j:
        p_val_df.iloc[i, j] = 0.0
      else:
        if method == "pearson":
          _, p_val = pearsonr(df.iloc[:, i], df.iloc[:, j])
        elif method == "kendall":
          _, p_val = kendalltau(df.iloc[:, i], df.iloc[:, j])
        p_val_df.iloc[i, j] = p_val

  return p_val_df


def show_heatmap(corr, title):
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        corr,
        annot=True,
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        fmt=".2f",
        linewidths=0.5,
    )
    
    plt.title(title)
    plt.show()


def show_scatterplot(df, x, y, xlabel: None, ylabel: None, title):
    xlabel = xlabel or x
    ylabel = ylabel or y

    sns.set_theme(style="whitegrid")
    
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        data=df,
        x=x,
        y=y,
        palette="deep",
        s=100,
    )
    
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

##### Regression

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

def vif_table(df, predictors):
    sub = df[predictors].dropna()
    X = sm.add_constant(sub).values
    cols = ['const'] + predictors
    vifs = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    return pd.DataFrame({'variable': cols, 'VIF': vifs}).iloc[1:]


def run_ols(df, predictors, target):
    sub = df[predictors + [target]].dropna()
    X = sm.add_constant(sub[predictors])
    y = sub[target]
    model = sm.OLS(y, X).fit()
    return model, sub


def loocv_report(df, predictors, target):
    model, sub = run_ols(df, predictors, target)
    infl = model.get_influence()

    h         = infl.hat_matrix_diag      
    loo_resid = infl.resid_press          
    y         = sub[target].values
    n, p      = int(model.nobs), model.df_model + 1

    press  = float((loo_resid ** 2).sum())
    sst    = float(((y - y.mean()) ** 2).sum())
    q2     = 1 - press / sst               
    rmse   = np.sqrt(press / n)
    mae    = np.abs(loo_resid).mean()
    y_hat  = y - loo_resid                 

    return {
        'target':   target,
        'n':        n,
        'R2_in':    model.rsquared,
        'adjR2_in': model.rsquared_adj,
        'Q2':       q2,
        'RMSE_cv':  rmse,
        'MAE_cv':   mae,
        'PRESS':    press,
    }, sub, y, y_hat, h, loo_resid, p


def coef_stability(df, predictors, target):
    _, sub = run_ols(df, predictors, target)
    X, y = sm.add_constant(sub[predictors]), sub[target]
    full = sm.OLS(y, X).fit().params
    loo  = pd.DataFrame(
        [sm.OLS(y.drop(y.index[i]), X.drop(X.index[i])).fit().params
         for i in range(len(sub))]
    )
    return pd.DataFrame({
        'full':       full,
        'loo_min':    loo.min(),
        'loo_max':    loo.max(),
        'loo_std':    loo.std(),
        'sign_flips': (np.sign(loo) != np.sign(full)).sum(),
    })


def loocv_summary(df, targets, features):
    summary_rows = []
    for tgt in targets:
        metrics, sub, y, y_hat, h, loo_resid, p = loocv_report(df, features, tgt)
        summary_rows.append(metrics)

        thr = 2 * p / metrics['n']
        flagged = sub.index[h > thr].tolist()

        print(f"\n--- {tgt} ---")
        print(f"R2_in={metrics['R2_in']:.3f}  Q2={metrics['Q2']:.3f}  "
            f"RMSE_cv={metrics['RMSE_cv']:.3f}  MAE_cv={metrics['MAE_cv']:.3f}")
        if flagged:
            print(f"  high-leverage (h>{thr:.2f}): {flagged}")

        pd.DataFrame({'observed': y, 'loo_pred': y_hat, 'leverage': h},
                    index=sub.index).to_csv(f'results/loocv_{tgt}.csv')

    summary = pd.DataFrame(summary_rows).set_index('target').round(3)
    return summary

### *Group 1*

In [ ]:
columns = ["name", "complexity_avg_raw_std", "complexity_avg_norm_std", 
           "complexity_div_std", "complexity_disp_std",  "complexity_diss_aa_seq_std",
           "f_score_imtf", "f_score_imts", "f_score_imto", 
           "f_score_imftf", "f_score_imfts", "f_score_imfto"]

# real life data
path = ROOT / "results" / "evaluation" / "real_life"
rl_complexity_path = path / "complexity.csv"
rl_quality_path = path / "quality.csv"

rl_c_df = pd.read_csv(rl_complexity_path)
rl_q_df = pd.read_csv(rl_quality_path)

rl_df_merge = pd.merge(rl_c_df, rl_q_df, on="name", how="inner")
rl_df = rl_df_merge[columns]

# synthetic data
path = ROOT / "results" / "evaluation" / "synthetic"
sy_complexity_path = path / "complexity.csv"
sy_quality_path = path / "quality.csv"

sy_c_df = pd.read_csv(sy_complexity_path)
sy_q_df = pd.read_csv(sy_quality_path)

sy_df_merge = pd.merge(sy_c_df, sy_q_df, on="name", how="inner")
sy_df = sy_df_merge[columns]

# complete dataset
rl_sy_df = pd.concat([rl_df, sy_df], axis=0, ignore_index=True)

,name,complexity_avg_raw_std,complexity_avg_norm_std,complexity_div_std,complexity_disp_std,complexity_diss_aa_seq_std,f_score_imtf,f_score_imts,f_score_imto,f_score_imftf,f_score_imfts,f_score_imfto
0,BPIC 12 A,-0.154923,-0.167488,1.286198,2.015005,1.808069,0.539207,0.539207,0.243423,0.689196,0.689196,0.416228
1,BPIC 12 O,-0.622845,0.157384,1.028602,2.691605,1.643130,0.826087,0.826087,0.575758,0.826087,0.826087,0.775510
2,BPIC 12 W,-1.024360,-0.926246,1.028602,1.017759,0.712468,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3,BPIC 13 Closed Problems,-0.518436,2.165405,0.169947,0.628173,-0.077836,0.857737,0.962351,0.857737,0.807151,0.925432,0.807151
4,BPIC 13 Incidents,-0.578992,0.799547,1.028602,1.667836,3.164910,0.814815,0.936170,0.814815,0.814815,0.936170,0.814815
5,BPIC 13 Open Problems,-0.359729,2.940092,-1.547361,-0.960494,-0.893695,0.791569,0.791569,0.791569,0.791569,0.791569,0.791569
6,BPIC 17 Offer,-0.744336,-0.654216,-0.903370,-0.673724,-0.713484,0.960763,0.960763,0.960763,0.960763,0.960763,0.960763
7,BPIC 19 2-way Match,-0.734832,-0.640961,-0.516976,-0.434249,-0.425220,0.903704,0.903704,0.903704,0.903704,0.903704,0.903704
8,BPIC 19 Consignment,0.413875,0.123951,1.672592,-0.282127,-0.308110,0.751893,0.868701,0.751893,0.869847,0.869206,0.869847
9,BPIC 20 Domestic Declarations,-0.380806,-0.581323,-1.179366,-0.750011,-0.677702,0.980356,1.000000,0.980356,0.753246,1.000000,0.753246


### *Group 2*

In [ ]:
# simulated data
path = ROOT / "results" / "evaluation" / "simulated"
sm_complexity_path = path / "complexity.csv"
sm_quality_path = path / "quality.csv"

sm_c_df = pd.read_csv(sm_complexity_path)
sm_q_df = pd.read_csv(sm_quality_path)

sm_df_merge = pd.merge(sm_c_df, sm_q_df, on="name", how="inner")
sm_df = sm_df_merge[columns]

### *Analysis*